# Places POI Pull — Illicit Pharmacy Indicators

Performs a systematic grid-based search across Gauteng and KwaZulu-Natal using the Google Places Nearby Search API to capture points of interest that could indicate unregistered or illicit pharmacy activity.

Each province is covered by a grid of search points spaced ~8km apart with a 5km search radius per point, ensuring full coverage with overlapping circles. Only grid points that fall inside the province boundary are queried.

Six keywords are searched in sequence. Places that appear under multiple keywords are merged into a single row with a combined `keyword` field rather than being duplicated.

Keywords: `pharmacy`, `muti shop`, `chemist`, `health shop`, `traditional medicine`, `medicine shop`

Output: `poi_pharmacies_google.csv`

## Imports and configuration

In [ ]:
import time
import requests
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely import wkt
from shapely.geometry import Point

API_KEY = "your-api-key"

SEARCH_RADIUS = 5000   # meters — 5km per grid point
GRID_SPACING  = 0.08   # degrees — ~8km spacing; overlapping radii ensure no gaps

# All keywords to search. Each is run separately and results merged by place_id.
KEYWORDS = [
    "pharmacy",
    "muti shop",
    "chemist",
    "health shop",
    "traditional medicine",
    "medicine shop",
]

PROVINCE_BOUNDARY_DIR = r"C:\Users\jcahi\Dropbox\PC\Downloads"
OUTPUT_PATH = r"C:\Users\jcahi\Dropbox\PC\Downloads\poi_pharmacies_google.csv"

## Load province boundaries and define grid generator

In [ ]:
gauteng_csv = pd.read_csv(f"{PROVINCE_BOUNDARY_DIR}\\gauteng_boundaries_geojson.csv")
kzn_csv     = pd.read_csv(f"{PROVINCE_BOUNDARY_DIR}\\kzn_boundaries_geojson.csv")

# Reconstruct GeoDataFrames from WKT geometry column
gauteng = gpd.GeoDataFrame(gauteng_csv, geometry=gauteng_csv['geometry'].apply(wkt.loads), crs='EPSG:4326')
kzn     = gpd.GeoDataFrame(kzn_csv,     geometry=kzn_csv['geometry'].apply(wkt.loads),     crs='EPSG:4326')

print(f"Gauteng loaded : {len(gauteng)} polygon(s)")
print(f"KZN loaded     : {len(kzn)} polygon(s)")


def generate_grid(province_gdf):
    # Build a regular lat/lng grid covering the province bounding box,
    # then keep only points that fall inside the actual province boundary
    bounds         = province_gdf.total_bounds  # minx, miny, maxx, maxy
    province_shape = province_gdf.union_all()

    points = [
        (lat, lng)
        for lat in np.arange(bounds[1], bounds[3], GRID_SPACING)
        for lng in np.arange(bounds[0], bounds[2], GRID_SPACING)
        if province_shape.contains(Point(lng, lat))
    ]

    print(f"  Generated {len(points):,} grid points")
    return points

## Cost estimate

Run this before executing the full search to confirm API spend is acceptable. Each grid point can trigger up to 3 API calls (one per page of 20 results). Nearby Search is billed at $32 per 1,000 calls.

In [ ]:
gauteng_points = generate_grid(gauteng)
kzn_points     = generate_grid(kzn)

total_points   = len(gauteng_points) + len(kzn_points)
n_keywords     = len(KEYWORDS)

print(f"Gauteng grid points : {len(gauteng_points):,}")
print(f"KZN grid points     : {len(kzn_points):,}")
print(f"Total grid points   : {total_points:,}")
print(f"Keywords to search  : {n_keywords}")
print()

# Cost range across all keywords
# Best case: 1 API call per point (< 20 results, no pagination)
# Worst case: 3 API calls per point (60 results across 3 pages)
best_case  = (total_points * n_keywords * 1 * 32) / 1000
worst_case = (total_points * n_keywords * 3 * 32) / 1000
print(f"Estimated cost range: ${best_case:.2f} – ${worst_case:.2f}")

## API and parsing functions

`search_nearby` handles a single grid point for a given keyword, including pagination via `next_page_token` and exponential backoff on API errors.

`parse_results` flattens the raw API response list into a DataFrame.

`pull_keyword_for_province` runs the full grid search for one province and one keyword, deduplicating on `place_id` within the run.

In [ ]:
def search_nearby(lat, lng, keyword, retries=3):
    # Queries Nearby Search at (lat, lng) and follows pagination until exhausted
    url     = "https://maps.googleapis.com/maps/api/place/nearbysearch/json"
    params  = {"location": f"{lat},{lng}", "radius": SEARCH_RADIUS,
                "keyword": keyword, "key": API_KEY}
    results = []

    while True:
        for attempt in range(retries):
            response = requests.get(url, params=params)
            data     = response.json()
            if data.get("status") in ("OK", "ZERO_RESULTS"):
                break
            wait = 2 ** attempt
            print(f"    Retrying in {wait}s (attempt {attempt + 1}/{retries})...")
            time.sleep(wait)
        else:
            print(f"  API error: {data.get('status')} at {lat},{lng}")
            break

        results.extend(data.get("results", []))

        next_page = data.get("next_page_token")
        if next_page:
            time.sleep(2)   # Google requires a short delay before the next-page token activates
            params = {"pagetoken": next_page, "key": API_KEY}
        else:
            break

    return results


def parse_results(results, province_name, keyword):
    rows = []
    for r in results:
        rows.append({
            "place_id":           r.get("place_id"),
            "name":               r.get("name"),
            "address":            r.get("vicinity"),
            "lat":                r["geometry"]["location"]["lat"],
            "lng":                r["geometry"]["location"]["lng"],
            "rating":             r.get("rating"),
            "user_ratings_total": r.get("user_ratings_total"),
            "business_status":    r.get("business_status"),
            "types":              ", ".join(r.get("types", [])),
            "province":           province_name,
            "keyword":            keyword,
        })
    return pd.DataFrame(rows)


def pull_keyword_for_province(province_gdf, province_name, keyword):
    print(f"  {province_name} | '{keyword}'")
    grid_points = generate_grid(province_gdf)
    all_results = {}   # keyed by place_id to deduplicate within this search pass

    for i, (lat, lng) in enumerate(grid_points):
        results = search_nearby(lat, lng, keyword)
        for r in results:
            all_results[r['place_id']] = r
        if (i + 1) % 50 == 0:
            print(f"    {i + 1:,} / {len(grid_points):,} points searched | {len(all_results):,} unique so far")
        time.sleep(0.5)

    print(f"    Done — {len(all_results):,} unique results")
    return list(all_results.values())


print("Functions defined.")

## Run all keyword searches

Iterates every keyword across both provinces. Results are accumulated in memory and merged at the end — the CSV is not written until all searches are complete.

Places that appear under multiple keywords are collapsed into a single row with a combined `keyword` field (e.g. `chemist, pharmacy`).

In [ ]:
all_frames = []

for keyword in KEYWORDS:
    print(f"\nKeyword: '{keyword}'")

    for province_gdf, province_name in [(gauteng, "Gauteng"), (kzn, "KwaZulu-Natal")]:
        raw     = pull_keyword_for_province(province_gdf, province_name, keyword)
        df      = parse_results(raw, province_name, keyword)
        all_frames.append(df)

print("\nAll searches complete.")

## Merge keywords and save

Groups by `place_id` and collapses duplicate records, combining all keyword tags into a single comma-separated field. This preserves the information that a location appeared under multiple search terms.

In [ ]:
combined = pd.concat(all_frames, ignore_index=True)
print(f"Total records before dedup : {len(combined):,}")

# Aggregate fields — non-keyword fields take the first non-null value per place_id
combined = (
    combined
    .groupby('place_id', as_index=False)
    .agg({
        'name':               'first',
        'address':            'first',
        'lat':                'first',
        'lng':                'first',
        'rating':             'first',
        'user_ratings_total': 'first',
        'business_status':    'first',
        'types':              'first',
        'province':           'first',
        'keyword':            lambda x: ', '.join(sorted(set(x.dropna()))),
    })
)

print(f"Total records after dedup  : {len(combined):,}")
print(f"Removed duplicates         : {len(pd.concat(all_frames)) - len(combined):,}")
print()

multi = combined[combined['keyword'].str.contains(',')]
print(f"Places tagged under multiple keywords: {len(multi):,}")
print()
print("Province breakdown:")
print(combined['province'].value_counts().to_string())
print()
print("Keyword breakdown (multi-tagged counted once per tag):")
tag_counts = (
    combined['keyword']
    .str.split(', ')
    .explode()
    .value_counts()
)
print(tag_counts.to_string())

combined.to_csv(OUTPUT_PATH, index=False)
print(f"\nSaved {len(combined):,} records -> {OUTPUT_PATH}")